In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

In [20]:
registry_df = pd.read_csv('current_rental_registrations_251001.csv')
violations_df = pd.read_csv('Building-Property-Violations.csv')

In [21]:
# --- Step 2: Clean and Standardize the SAM ID Columns ---
# For the violations dataset, the column is 'sam_id'
violations_df['sam_id'] = violations_df['sam_id'].astype(str).str.strip().str.lower()
# Replace 'nan' string with actual None
violations_df['sam_id'] = violations_df['sam_id'].replace({'nan': None})

# For the registry dataset, the column is 'SAMid'
registry_df['SAMid'] = registry_df['SAMid'].astype(str).str.strip().str.lower()
registry_df['SAMid'] = registry_df['SAMid'].replace({'nan': None})
# Rename 'SAMid' to 'sam_id' for consistency
registry_df = registry_df.rename(columns={'SAMid': 'sam_id'})

# Drop duplicate rows in both datasets (if any)
violations_df = violations_df.drop_duplicates()
registry_df = registry_df.drop_duplicates()

In [22]:
violations_df

,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V91983,1013,NaN,Closed,121.2,NaN,Unsafe and Dangerous,302,NaN,Sumner,ST,East Boston,MA,02128,01,302 Sumner St,NaN,East Boston,MA,02128,132380.0,42.367678,-71.036580,"(42.367678491254956, -71.0365803778755)"
1,V814146,1013,2025-02-12 09:27:00,Open,102.8,NaN,Maintenance,1509,1517,Blue Hill,AV,Mattapan,MA,02126,18,17 Brook Rd,NaN,Milton,MA,02186,16298.0,42.272380,-71.093891,"(42.27237957706929, -71.09389071876852)"
2,V815295,1013,2025-02-12 08:41:47,Open,105.1,NaN,Failure to Obtain Permit,922,NaN,Canterbury,ST,Roslindale,MA,02131,18,126 COMMONWEALTH AV,NaN,DEDHAM,MA,02026,25538.0,42.278980,-71.115961,"(42.27897960699359, -71.11596075443885)"
3,V815283,1013,2025-02-12 08:07:06,Open,116,NaN,Unsafe Structures,43,NaN,Woolson,ST,Mattapan,MA,02126,14,84 Bess Road,NaN,Needham,MA,02492,152140.0,42.281320,-71.090121,"(42.28131957023638, -71.09012069132322)"
4,V755522,1013,2025-02-11 12:13:46,Open,1001.3.2,NaN,Testing & Certification,1273,NaN,Massachusetts,AV,Dorchester,MA,02125,07,1273 MASSACHUSETTS AVE,NaN,DORCHESTER,MA,02125,91955.0,42.321360,-71.062561,"(42.32135952831996, -71.06256053855205)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16492,V53371,1013,2009-12-03 11:26:33,Closed,103,NaN,Maintenance,55,NaN,Fenwood,RD,Boston,MA,02115,10,2 NEW WHITNEY ST,NaN,BOSTON,MA,02115,57589.0,42.335320,-71.107611,"(42.33531958780805, -71.10761060854603)"
16493,V53076,1013,2009-12-03 10:01:21,Closed,110.1,NaN,Inspections,225,NaN,Chelsea,ST,East Boston,MA,02128,01,45 FALCON ST,NaN,EAST BOSTON,MA,02128,30417.0,42.375199,-71.032310,"(42.37519948558022, -71.03231035067552)"
16494,V53175,1013,2009-12-02 11:32:46,Closed,110.1,NaN,Inspections,2,,French,TE,ROXBURY,MA,02120,10,2 FRENCH TER,NaN,ROXBURY CROSSING,MA,02120,None,42.330640,-71.097051,"(42.330639573986325, -71.09705059581793)"
16495,V53078,1013,2009-12-01 13:39:34,Closed,110.1,NaN,Inspections,1670,NaN,Washington,ST,Roxbury,MA,02118,08,1670 WASHINGTON ST,NaN,BOSTON,MA,02118,144315.0,42.337380,-71.075271,"(42.33737954464646, -71.07527053243535)"


In [23]:
violations_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16481 entries, 0 to 16496
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   case_no           16481 non-null  object 
 1   ap_case_defn_key  16481 non-null  int64  
 2   status_dttm       16480 non-null  object 
 3   status            16481 non-null  object 
 4   code              16481 non-null  object 
 5   value             0 non-null      float64
 6   description       16234 non-null  object 
 7   violation_stno    16481 non-null  object 
 8   violation_sthigh  4162 non-null   object 
 9   violation_street  16481 non-null  object 
 10  violation_suffix  16342 non-null  object 
 11  violation_city    16481 non-null  object 
 12  violation_state   16481 non-null  object 
 13  violation_zip     16479 non-null  object 
 14  ward              16481 non-null  object 
 15  contact_addr1     16476 non-null  object 
 16  contact_addr2     2956 non-null   object 
 17

In [24]:
violations_drop = ['ap_case_defn_key', 'value', 'violation_sthigh', 'contact_addr2']
violations_df = violations_df.drop(violations_drop, axis=1)
violations_df

,case_no,status_dttm,status,code,description,violation_stno,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V91983,NaN,Closed,121.2,Unsafe and Dangerous,302,Sumner,ST,East Boston,MA,02128,01,302 Sumner St,East Boston,MA,02128,132380.0,42.367678,-71.036580,"(42.367678491254956, -71.0365803778755)"
1,V814146,2025-02-12 09:27:00,Open,102.8,Maintenance,1509,Blue Hill,AV,Mattapan,MA,02126,18,17 Brook Rd,Milton,MA,02186,16298.0,42.272380,-71.093891,"(42.27237957706929, -71.09389071876852)"
2,V815295,2025-02-12 08:41:47,Open,105.1,Failure to Obtain Permit,922,Canterbury,ST,Roslindale,MA,02131,18,126 COMMONWEALTH AV,DEDHAM,MA,02026,25538.0,42.278980,-71.115961,"(42.27897960699359, -71.11596075443885)"
3,V815283,2025-02-12 08:07:06,Open,116,Unsafe Structures,43,Woolson,ST,Mattapan,MA,02126,14,84 Bess Road,Needham,MA,02492,152140.0,42.281320,-71.090121,"(42.28131957023638, -71.09012069132322)"
4,V755522,2025-02-11 12:13:46,Open,1001.3.2,Testing & Certification,1273,Massachusetts,AV,Dorchester,MA,02125,07,1273 MASSACHUSETTS AVE,DORCHESTER,MA,02125,91955.0,42.321360,-71.062561,"(42.32135952831996, -71.06256053855205)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16492,V53371,2009-12-03 11:26:33,Closed,103,Maintenance,55,Fenwood,RD,Boston,MA,02115,10,2 NEW WHITNEY ST,BOSTON,MA,02115,57589.0,42.335320,-71.107611,"(42.33531958780805, -71.10761060854603)"
16493,V53076,2009-12-03 10:01:21,Closed,110.1,Inspections,225,Chelsea,ST,East Boston,MA,02128,01,45 FALCON ST,EAST BOSTON,MA,02128,30417.0,42.375199,-71.032310,"(42.37519948558022, -71.03231035067552)"
16494,V53175,2009-12-02 11:32:46,Closed,110.1,Inspections,2,French,TE,ROXBURY,MA,02120,10,2 FRENCH TER,ROXBURY CROSSING,MA,02120,None,42.330640,-71.097051,"(42.330639573986325, -71.09705059581793)"
16495,V53078,2009-12-01 13:39:34,Closed,110.1,Inspections,1670,Washington,ST,Roxbury,MA,02118,08,1670 WASHINGTON ST,BOSTON,MA,02118,144315.0,42.337380,-71.075271,"(42.33737954464646, -71.07527053243535)"


In [26]:
registry_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40041 entries, 0 to 40040
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LICENSENO                   40041 non-null  object 
 1   sam_id                      39911 non-null  object 
 2   row_number_licenseno_SAMid  39818 non-null  float64
 3   RegisteredAddress           40041 non-null  object 
 4   LICENSECAT                  40041 non-null  object 
 5   OWNEROCC                    39604 non-null  object 
 6   NUMBEROFUNITS               40041 non-null  int64  
 7   TOTALUNITS                  40041 non-null  int64  
 8   SystemAddDate               40041 non-null  object 
 9   IssueDate                   39475 non-null  object 
 10  NextRenewal                 40041 non-null  object 
 11  ApplicationDate             40038 non-null  object 
 12  ExpirationDate              40041 non-null  object 
 13  Milestone                   400

In [27]:
# --- Step 3: Separate Records with Missing SAM IDs ---
# Rows in violations with missing sam_id
missing_sam_violations = violations_df[violations_df['sam_id'].isnull()]
# Rows in registry with missing sam_id
missing_sam_registry = registry_df[registry_df['sam_id'].isnull()]
# Combine missing SAM id records from both datasets
#missing_sam_combined = pd.concat([missing_sam_violations, missing_sam_registry], ignore_index=True)


In [28]:
# --- Step 4: Create DataFrame for Unregistered Properties ---
# Filter valid records (i.e. where sam_id is not missing)
valid_violations = violations_df[violations_df['sam_id'].notnull()]
valid_registry = registry_df[registry_df['sam_id'].notnull()]

# Build a set of registered sam_ids from the registry dataset
registered_ids = set(valid_registry['sam_id'])
# Identify unregistered properties: those in violations but NOT in registry
unregistered_properties = valid_violations[~valid_violations['sam_id'].isin(registered_ids)]

In [9]:
print("Unregistered Properties (in violations but not in registry):")
unregistered_properties

Unregistered Properties (in violations but not in registry):


,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
1,V814146,1013,2025-02-12 09:27:00,Open,102.8,NaN,Maintenance,1509,1517,Blue Hill,AV,Mattapan,MA,02126,18,17 Brook Rd,NaN,Milton,MA,02186,16298.0,42.272380,-71.093891,"(42.27237957706929, -71.09389071876852)"
2,V815295,1013,2025-02-12 08:41:47,Open,105.1,NaN,Failure to Obtain Permit,922,NaN,Canterbury,ST,Roslindale,MA,02131,18,126 COMMONWEALTH AV,NaN,DEDHAM,MA,02026,25538.0,42.278980,-71.115961,"(42.27897960699359, -71.11596075443885)"
5,V809952,1013,2025-02-11 12:13:02,Open,105.1,NaN,Failure to Obtain Permit,70,NaN,Commonwealth,AV,Boston,MA,02116,05,70 COMMONWEALTH AVE,NaN,BOSTON,MA,02116,41226.0,42.352320,-71.074890,"(42.352319541980954, -71.07489049713848)"
6,V815076,1013,2025-02-11 10:39:39,Open,102.8,NaN,Maintenance,15,NaN,Everdean,ST,Dorchester,MA,02122,16,15 Everdean ST,NaN,DORCHESTER,MA,02122,54890.0,42.300870,-71.050941,"(42.30086951407573, -71.05094055868085)"
7,V814118,1013,2025-02-07 10:25:12,Open,105.1,NaN,Failure to Obtain Permit,190,NaN,Springfield,ST,Roxbury,MA,02118,09,190 W Springfield St,NaN,Boston,MA,02118,129941.0,42.339940,-71.080101,"(42.3399395506349, -71.0801005364918)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16488,V53710,1013,2009-12-09 15:18:29,Closed,110.1,NaN,Inspections,23,NaN,Winship,ST,Brighton,MA,02135,22,28 Brooks Street,NaN,BRIGHTON,MA,02135,150792.0,42.347920,-71.151521,"(42.3479196433476, -71.15152067856192)"
16489,V53686,1013,2009-12-08 15:06:40,Closed,110.1,NaN,Inspections,125,NaN,Beaver,ST,Hyde Park,MA,02136,18,125 BEAVER ST,NaN,HYDE PARK,MA,02136,13016.0,42.256040,-71.131111,"(42.25603963224487, -71.13111084012345)"
16493,V53076,1013,2009-12-03 10:01:21,Closed,110.1,NaN,Inspections,225,NaN,Chelsea,ST,East Boston,MA,02128,01,45 FALCON ST,NaN,EAST BOSTON,MA,02128,30417.0,42.375199,-71.032310,"(42.37519948558022, -71.03231035067552)"
16495,V53078,1013,2009-12-01 13:39:34,Closed,110.1,NaN,Inspections,1670,NaN,Washington,ST,Roxbury,MA,02118,08,1670 WASHINGTON ST,NaN,BOSTON,MA,02118,144315.0,42.337380,-71.075271,"(42.33737954464646, -71.07527053243535)"


In [36]:
len(unregistered_properties)/len(violations_df)

0.7411564832231053

In [38]:
missing_sam_violations

,case_no,status_dttm,status,code,description,violation_stno,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
63,V808697,2025-01-21 09:57:00,Open,105.1,Failure to Obtain Permit,888,Morton,ST,Mattapan,MA,02126,17,619 Centre Street,Jamaica Plain,MA,02130,None,NaN,NaN,NaN
316,V790949,2024-09-30 11:56:11,Closed,105.1,Failure to Obtain Permit,1515,VFW Parkway,ST,BOSTON,MA,,20,1515 VFW PARKWAY #R77,WEST ROXBURY,ma,02132,None,NaN,NaN,NaN
392,V782808,2024-08-29 15:12:42,Closed,102.8,Maintenance,30,Washington,ST,Allston,MA,02135,21,30 WASHINGTON,BRIGHTON,MA,02135,None,34.244387,-73.651391,"(34.244386753001244, -73.6513909034731)"
1027,V723010,2024-01-02 11:15:34,Closed,102.8,Maintenance,135,CLARENDON,ST,BOSTON,MA,02116,04,135 Clarendon Street,Boston,MA,02116,None,34.244387,-73.651391,"(34.244386753001244, -73.6513909034731)"
1426,V679078,2023-06-07 09:23:27,Open,102.8,Maintenance,1875,Dorchester,AV,Dorchester,MA,02124,16,125 A Amory Street,Boston,MA,02119,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15284,V83334,2010-10-06 10:19:10,Closed,110.1,Inspections,1256,River,ST,Hyde Park,MA,02136,18,1340 Centre St #101,Newton,MA,02459,None,34.244387,-73.651391,"(34.244386753001244, -73.6513909034731)"
15402,V81281,2010-09-14 11:24:29,Closed,110.1,Inspections,1265,Boylston,ST,BOSTON,MA,02215,05,1265 Boylston St,Boston,MA,02115,None,42.345268,-71.096382,"(42.34526757153878, -71.09638156194613)"
15675,V73517,2010-07-16 09:04:25,Closed,110.11,Deteriorating Agents,427,Marlborough,ST,Boston,MA,02115,05,"Att: Jeff Berenbaum, 252 Newbury St",Boston,MA,02116,None,NaN,NaN,NaN
16019,V65453,2010-05-06 11:33:50,Closed,5121,Unsafe Structures,,,,,MA,,,417 Meridian St,East Boston,MA,02128,None,NaN,NaN,NaN


In [39]:
missing_sam_registry

,LICENSENO,sam_id,row_number_licenseno_SAMid,RegisteredAddress,LICENSECAT,OWNEROCC,NUMBEROFUNITS,TOTALUNITS,SystemAddDate,IssueDate,NextRenewal,ApplicationDate,ExpirationDate,Milestone,cntctkey,CAPACITY,CONTACTTYPE,ISLICENSEE,contacttype,ApplicantLastName,ApplicantFirstName,ApplicantAddress,ApplicantAddress2,ApplicantCity,ApplicantState,ApplicantPhone,ApplicantEmail
0,Rent-104588,None,NaN,"54 Desoto RD , WEST ROXBURY, MA 02132",1-3FAM,N,2,2,2013-08-20 09:47:26.000,2022-03-10 12:39:42.000,2026-01-01 00:00:00.000,2013-08-20 09:47:27.000,2024-12-31 00:00:00.000,Renewal Pending,952178,,2,Y,AGENT,Sougarides,Peter,54 Desoto Road,NaN,West Roxbury,MA,(617)875-3666,psougarides@samuelsre.com
1,Rent-105479,None,NaN,"152H Chestnut AV , Jamaica Plain, MA 02130",1-3FAM,N,2,2,2013-08-23 21:18:24.000,2013-08-23 21:20:26.000,2026-01-01 00:00:00.000,2013-08-23 21:18:25.000,2024-12-31 00:00:00.000,Renewal Pending,1044887,,2,Y,HOME,Cronin,Jeff,"152 Chestnut Avenue, Apt. 1",NaN,Jamaica Plain,MA,(626)437-7957,clancy_101@msn.com
2,Rent-106123,None,NaN,"103 Ninth ST , Charlestown, MA 02129",MultiFam,N,112,112,2013-08-27 10:49:42.000,2013-10-15 15:22:41.000,2026-01-01 00:00:00.000,2013-08-27 10:49:42.000,2023-12-31 00:00:00.000,Renewal Pending,2393013,,3,N,OWNER,Barkan,Peter,7 Wells Avenue,Suite 11,Newton,MA,(617)532-8605,pbarkan@barkanco.com
3,Rent-106123,None,NaN,"103 Ninth ST , Charlestown, MA 02129",MultiFam,N,112,112,2013-08-27 10:49:42.000,2013-10-15 15:22:41.000,2026-01-01 00:00:00.000,2013-08-27 10:49:42.000,2023-12-31 00:00:00.000,Renewal Pending,2268799,,2,Y,,Moser,Hannah,103 Ninth Street,NaN,Charlestown,MA,(617)242-4515,manager.anchorageapartments@barkanmanagement.com
4,Rent-106166,None,NaN,", , MA",1-3FAM,Y,1,2,2013-08-27 11:22:58.000,2013-08-27 11:24:34.000,2026-01-01 00:00:00.000,2013-08-27 11:22:58.000,2024-12-31 00:00:00.000,Renewal Pending,1105511,,2,Y,,Camacho,Laylanni,19 Harlem St,NaN,Boston,MA,(617)953-2825,laylanni.camacho@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Rent-96537,None,NaN,"3-5 MT PLEASANT AV AV , BOSTON, MA 02119",1-3FAM,Y,2,3,2013-08-02 18:42:39.000,2013-08-02 18:49:46.000,2026-01-01 00:00:00.000,2013-08-02 18:42:41.000,2021-12-31 00:00:00.000,Renewal Pending,1087080,,2,Y,HOME,Cort,Tchad,96 Mount Pleasant Avenue,NaN,Boston,MA,6178885280,tchadcort@hotmail.com
126,Rent-96676,None,NaN,"1259-1261 Dorchester AV , DORCHESTER, MA 02122",1-3FAM,N,3,3,2013-08-04 23:38:19.000,2013-10-08 15:22:02.000,2026-01-01 00:00:00.000,2013-08-04 23:38:19.000,2024-12-31 00:00:00.000,Renewal Pending,876920,,2,Y,,Vu,Quang,111 Pierce Ave,NaN,Boston,MA,(857)939-0331,vumanagementco@gmail.com
127,Rent-96676,None,NaN,"1259-1261 Dorchester AV , DORCHESTER, MA 02122",1-3FAM,N,3,3,2013-08-04 23:38:19.000,2013-10-08 15:22:02.000,2026-01-01 00:00:00.000,2013-08-04 23:38:19.000,2024-12-31 00:00:00.000,Renewal Pending,1087163,,3,N,OWNER,Vu,Quang,330 ADAMS STREET,NaN,QUINCY,MA,(857)939-0331,mariatvu@gmail.com
128,Rent-96810,None,NaN,", , MA",1-3FAM,Y,1,2,2013-08-05 15:17:58.000,2013-08-05 15:22:18.000,2026-01-01 00:00:00.000,2013-08-05 15:17:58.000,2021-12-31 00:00:00.000,Renewal Pending,1087544,,2,Y,APPL,Rodriguez,Joshua,82 Callender Street,NaN,Dorchester,MA,(617)584-5232,glenda45@verizon.net


In [ ]:
#exploring fuzzymatching for the properties without sam_id